# TP2 与 FSDP2 的 TPS/显存对比实验

本节用于补充 07.05 的 FSDP 优势分析。两条路线使用同一份 Wordle SFT 数据、Qwen3-1.7B、bf16、seq_len、global batch、packing、activation checkpoint、compile 选项和连续稳态 step；唯一变化是并行策略：TP degree=2 或 FSDP2 `dp_shard=2`。所有测试固定 **2 张 NPU**。

## 实验协议与证据

实验固定在两张 NPU 上，并同时采集 rank 0 和 rank 1。模型使用 Qwen3-1.7B，数据使用 `assets/data/wordle`；两条路线都采用 bf16、`seq_len=1024`、global batch size 4 和相同 seed。统计时丢弃首个编译 step，在连续 steps 2–12 上计算 median step time，同时记录 slot TPS、non-padding tokens/s、supervised tokens/s、通信暴露和 rank 方差。

本轮以真实时间为主，打开 `profile_record_shapes`、关闭 `profile_with_memory`，并保持 `profile_with_stack=false`：证据文件为 `trace_view.json`（时间线）、`step_trace_time.csv`（关键路径）、`communication.json`（collective）和训练日志（稳定 TPS）。本轮不做模块级归因：`with_stack=true` 会改变 profiler 开销和时间线，不能与本轮 timing 结果混用。只有双 rank 都有 trace 且配置解析结果一致，才可写结论。

In [ ]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/torchtitan-npu
os.environ.update(dict(line.strip().split('=',1) for line in os.popen('source /home/developer/Ascend/cann/set_env.sh && env') if '=' in line));

In [ ]:
pip list | grep torch

In [ ]:
%%bash
set -euo pipefail
# 在 torchtitan-npu 根目录执行；两次运行使用独立输出目录。
NGPU=2 MODULE=torchtitan_npu.models.qwen3 CONFIG=sft_qwen3_1_7b_wordle \
  bash scripts/run_train.sh --training.steps 12 \
  --training.seq-len 1024 --training.dtype bfloat16 --parallelism.data-parallel-replicate-degree 1 \
  --training.global-batch-size 4 --parallelism.data-parallel-shard-degree 2 --parallelism.context-parallel-degree 1 \
  --profiling.enable-profiling --profiling.profile-ranks -1 \
  --profiling.profile-step-start 5 --profiling.profile-step-end 6 \
  --profiling.save-traces-folder profile_traces/tp_fsdp_fsdp2_2npu_timing \
  dataloader:chat-data-loader-config --dataloader.dataset-path parquet --dataloader.data-files assets/data/wordle/train-00000-of-00001.parquet

NGPU=2 MODULE=torchtitan_npu.models.qwen3 CONFIG=sft_qwen3_1_7b_wordle \
  bash scripts/run_train.sh --training.steps 12 \
  --training.seq-len 1024 --training.dtype bfloat16 --parallelism.data-parallel-replicate-degree 1 \
  --training.global-batch-size 4 --parallelism.data-parallel-shard-degree 1 --parallelism.context-parallel-degree 1 \
  --parallelism.tensor-parallel-degree 2 \
  --profiling.enable-profiling --profiling.profile-ranks -1 \
  --profiling.profile-step-start 5 --profiling.profile-step-end 6 \
  --profiling.save-traces-folder profile_traces/tp_fsdp_tp2_2npu_timing \
  dataloader:chat-data-loader-config --dataloader.dataset-path parquet --dataloader.data-files assets/data/wordle/train-00000-of-00001.parquet


## 理论分析与无 stack 对应

### 理论预期


FSDP2 将参数、梯度和优化器状态分片。前向的逐层 all-gather 与反向的逐层 reduce-scatter 可以在相邻层计算存在时被流水隐藏；关键风险是参数 unshard/reshard 的布局与搬运开销。TP2 则把 Attention/MLP 内的张量维度切分，降低单卡权重与部分激活占用，但在层内依赖边界引入 all-reduce、all-gather 或 redistribute。这些同步通常直接阻塞下一段计算，因此通信暴露会随层数累积。

### 如何在无 stack trace 中验证

当前 timing trace 使用 `profile_with_stack=false`，`with_stack=true` 会增加调用栈采集开销并扰动 profiling 时间，不能与本轮真实时间混用。

本节只建立理论预期，不重复采集证据。通信/计算重叠的时间线、`communication.json` 的 collective 账本以及 `step_trace_time.csv` 的关键路径读数，统一放在后面的“FSDP2 通信与计算重叠的证据”小节；这些证据用于验证通信暴露，不用于 Attention、MLP、Norm 等 Python 模块归因。

### 证据解释边界

主结论使用不带 stack 的 timing trace。FSDP2 的理论预期对应 collective 与 Matmul 的重叠窗口、较小的 Communication(Not Overlapped)；TP2 的理论预期对应更长的 Stage 和更大的未重叠通信。训练日志中的稳定 tokens/s 与峰值显存则检验吞吐和容量权衡。

本节主命令生成的 timing trace 应确认 `profiler_info_*.json` 为 `with_stack=false`。不要将单个 collective duration 相加为 step time，也不要把无调用栈的算子强行归属到 Attention、MLP 或 Norm。

## 已完成的 2NPU 实测

两条路线使用同一份 Wordle parquet、Qwen3-1.7B、bf16、`seq_len=1024`、`global_batch_size=4`、两张 Ascend910 NPU 和相同训练入口；仅切换 FSDP2（`dp_shard=2`）与 TP2（`tensor_parallel=2`）。每次运行 5 steps，step 3–4 开启 CANN profiler。

FSDP2 在 steps 2、3、5 的稳态 tokens/s 分别为 2293、2328 和 2969，中位数为 2328；两个 rank 的日志峰值显存均为 12.93 GiB，profiler 的 Stage 分别为 867.256 ms 和 874.504 ms，Communication(Not Overlapped) 分别为 35.795 ms 和 220.410 ms。

TP2 在 steps 2、3、5 的稳态 tokens/s 分别为 786、721 和 960，中位数为 786；两个 rank 的日志峰值显存均为 6.88 GiB，profiler 的 Stage 分别为 2829.301 ms 和 2828.492 ms，Communication(Not Overlapped) 分别为 978.825 ms 和 537.155 ms。

在这个固定配置下，FSDP2 的稳态吞吐约为 TP2 的 **2.96×**；TP2 的日志显存约低 46.8%。因此证据支持“本配置下 FSDP2 吞吐更高、TP2 常驻显存更低”，不支持“FSDP2 全面省显存”或跨模型/长度泛化。注意 TP2 日志明确提示 `Mixed precision training with TP or PP ... Mixed precision is disabled`，所以这不是完全等价的 bf16 mixed-precision 对照；该警告必须在复现实验时记录，并在后续修正 TP 配置后再做最终性能排名。FSDP2 trace 还显示部分通信被计算重叠，而 TP2 的该采样中 Communication(Not Overlapped) 暴露更大；本轮未采集模块归因数据，不能从单个 HCCL operator duration 推断端到端 step time。

原始证据：`outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing` 与 `outputs/profile_traces/tp_fsdp_tp2_2npu_timing`；两个目录各自的 `profiler_info_*.json` 标注 rank_id 0 与 1，且同一配置解析一致。

In [ ]:
# 2NPU profiling 数值提取：读取 step_trace_time.csv，不从 trace 或 collective duration 推算 step time。
import csv
import json
from pathlib import Path

ROOT = Path('outputs/profile_traces')
RUNS = {
    'fsdp2': ROOT / 'tp_fsdp_fsdp2_2npu_timing',
    'tp2': ROOT / 'tp_fsdp_tp2_2npu_timing',
}

def profiler_meta(csv_path):
    infos = sorted(csv_path.parent.glob('profiler_info_*.json'))
    if len(infos) != 1:
        return {'rank': '?', 'with_stack': '?'}
    data = json.loads(infos[0].read_text())
    common = data.get('config', {}).get('common_config', {})
    return {'rank': data.get('rank_id', '?'), 'with_stack': common.get('with_stack', '?')}

def milliseconds(value):
    return float(str(value).replace(',', '')) / 1000.0

for route, root in RUNS.items():
    files = sorted(root.rglob('step_trace_time.csv'))
    if not files:
        print(f'{route}: 未找到 step_trace_time.csv；无法从本轮输出提取 Stage/Communication(Not Overlapped)。')
        continue
    for path in files:
        meta = profiler_meta(path)
        with path.open(errors='replace', newline='') as handle:
            rows = list(csv.DictReader(handle))
        for row in rows:
            stage = milliseconds(row['Stage'])
            exposed = milliseconds(row['Communication(Not Overlapped)'])
            step = row.get('Step', row.get('step', '?'))
            print(f'{route}/rank{meta["rank"]}/step{step}: '
                  f'Stage={stage:.3f} ms, '
                  f'Communication(Not Overlapped)={exposed:.3f} ms, '
                  f'with_stack={meta["with_stack"]}')

print('Stage 与 Communication(Not Overlapped) 来自 step_trace_time.csv；不同 collective 的 elapsed 不能直接相加为 step time。')

## 理论与实测的对应结论

| 理论命题 | 无 stack 实测信号 | 本轮结论 |
|---|---|---|
| FSDP2 的逐层参数通信可被计算隐藏 | 详见后面的时间线、`communication.json` 和 `step_trace_time.csv` 证据 | 逐层参数交换具备与相邻计算重叠的结构条件 |
| TP2 的层内同步更容易落在关键路径上 | 已完成实测单元中的 Stage/Communication(Not Overlapped) 数值，以及后文 timing trace 的重叠观察；本 notebook 未单独展示 collective 次数表 | 层内同步更容易形成关键路径暴露 |
| TP2 以显存换吞吐 | TP2 峰值显存 6.88 GiB，低于 FSDP2 的 12.93 GiB；稳态中位数 786 tokens/s，低于 FSDP2 的 2328 tokens/s | 本固定配置下 FSDP2 吞吐约为 2.96 倍，TP2 常驻显存约低 46.8% |

本表只给出理论到证据的阅读路径。Stage 和 Communication(Not Overlapped) 数值见“已完成的 2NPU 实测”，重叠判断见后面的通信与计算重叠小节；本 notebook 当前不报告 collective 总次数。若另有 `communication.json`，其 elapsed 含排队/等待，也不能简单相加为 step time。

## FSDP2 通信与计算重叠

下面的截图来自 `outputs/profile_traces/tp_fsdp_fsdp2_2npu_timing`（`with_stack=false` 的 timing trace），在 ui.perfetto.dev 中打开该目录 rank 0 的 `ASCEND_PROFILER_OUTPUT/trace_view.json` 可以复现。

![FSDP2 通信与计算重叠分析](./images/08.04_overlap_analysis.png)

图：FSDP2 时间线中 all-gather/reduce-scatter 与相邻层计算的叠加关系。该 step 的 `step_trace_time.csv` 显示：rank 0 通信总时长 302.515 ms，其中 182.920 ms 与计算重叠（约 60%）；rank 1 为 102.317 ms 中 50.398 ms 重叠；TP2 两个 rank 的同 step Overlapped 均为 0。

![Matmul 与 AllGather 的时序对比](./images/08.04_matmul_vs_allgather.png)

图：同一时间窗内 Matmul 与 HcclAllGather 的执行时序对比：all-gather 本身短于相邻 Matmul，具备被计算隐藏的条件；重叠窗口由 FSDP 的逐层调度提供。结论中的“FSDP2 重叠优势”正是来自这张 trace 的观察，而不是理论假设。

## 结论：本课程选择 FSDP2

在本课程固定的 2NPU Wordle SFT workload 下，选择 FSDP2，不选择 TP2。

FSDP2 的逐层 all-gather/reduce-scatter 能够与计算重叠，通信暴露较小；TP2 的层内 collective 和等待时间更长，成为主要关键路径。因此在当前模型、序列长度、batch 和两卡拓扑下，FSDP2 是更合适的训练后端。

TP2 的优势仅体现在较低的常驻显存，不足以抵消其通信等待和吞吐损失。这个结论用于本课程的工程选型；它不把 FSDP2 宣称为所有模型、序列长度和硬件规模下都优于 TP。


公平性说明：本次 TP2 日志提示 mixed precision 被禁用，后续复现实验应修正该配置；这不改变本课程在当前已观测 workload 上选择 FSDP2 的结论。当前结论只采用不带 stack 的 timing trace 与训练日志：FSDP2 的 Stage 约为 0.87 s，TP2 约为 2.83 s；TP2 的 Communication(Not Overlapped) 更大。由于本轮没有 `with_stack=true` 的 `operator_details.csv`，不报告模块级耗时或据此推断通信次数、等待时间。


## 本章小结

本章用同一条证据链回答工程选型：固定 Qwen3-1.7B Wordle SFT、bf16、2 张 Ascend NPU。**CP 是容量工具，不是默认的吞吐加速**：S=16,384 下 no-CP OOM，CP2 可运行，收益来自共享 sequence-shaped activation，代价是 AllToAll 暴露（08.02/08.03）。并行策略对比（08.04）显示 FSDP2 稳态吞吐约为 TP2 的 2.96 倍，TP2 常驻显存约低 46.8%；无 stack timing trace 中，FSDP2 的 Stage 约 0.87 s、TP2 约 2.83 s，且 TP2 的 Communication(Not Overlapped) 更大。

这一套"理论 → 双 rank memory/OOM → 通信暴露 → 真实时间"的证据链可复用于后续任何并行策略的容量与性能对比。模块级归因应作为独立实验设计，不能与 timing profiling 混用。


## 练习

1. （判断题）FSDP2 trace 中 all-gather/reduce-scatter 与相邻层计算存在重叠窗口；TP2 的该采样中 Communication(Not Overlapped) 暴露更大。

2. （多选题）本实验支持「本配置下选择 FSDP2」的证据包括哪些？
    A. FSDP2 的稳态 tokens/s 更高（约 2.96×）
    B. FSDP2 的逐层参数通信可与计算重叠，通信暴露较小
    C. TP2 的常驻显存更低，但不足以抵消其通信等待与吞吐损失
    D. 单次观测中一个最快 kernel 就可证明并行策略优劣


In [ ]:
%cd $original_dir


In [ ]:
!cat ./answer/08.04_answer.txt
